In [1]:
import os
iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')
RANDOMSEED=1727

In [2]:
import torch,random
import numpy as np
def random_seed(seed_value, use_cuda=False):
    np.random.seed(seed_value) # cpu vars
    torch.manual_seed(seed_value) # cpu vars
    random.seed(seed_value) # Python
    if use_cuda: 
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value) # gpu vars
        torch.backends.cudnn.deterministic = True  #needed
        torch.backends.cudnn.benchmark = False
random_seed(1727)

In [3]:
from pathlib import Path

cred_path = Path('~/.kaggle/access_token').expanduser()
if not cred_path.exists():
    cred_path.parent.mkdir(exist_ok=True)
    cred_path.write_text("KGAT_9f6b15aaf6f7637b8497dfb3c56c079e")
    cred_path.chmod(0o600)

In [4]:
path = Path('ml-olympiad-machine-translation-french-wolof')

In [5]:
if not iskaggle:
    if not path.exists():
        import zipfile,kaggle
        kaggle.api.competition_download_cli(str(path))
        zipfile.ZipFile(f'{path}.zip').extractall(path)
else:
    # /kaggle/input/competitions/ml-olympiad-machine-translation-french-wolof/train.csv
    path = Path('/kaggle/input/competitions/ml-olympiad-machine-translation-french-wolof')

# %pip install -q datasets
!dir /o:g /w {path}
# !ls {path}

 Volume in drive C is Windows
 Volume Serial Number is 6291-898F

 Directory of c:\Users\longnuub\learning-programming-languages\learning-python\kaggle\ml-olympiad-machine-translation-french-wolof

[..]        [.]         train.csv   test.csv    
               2 File(s)      2.371.904 bytes
               2 Dir(s)  131.712.831.488 bytes free


# loading the data

In [6]:
import pandas as pd
train=pd.read_csv(path/"train.csv")
test=pd.read_csv(path/"test.csv")

# drop the `id` col for training data ONLY, we need the id for test preds later
train.drop(columns=["ID"],inplace=True)

In [7]:
# print(train[:10])
train[:10]
# test.head()
# train.describe()

,FRENCH,WOLOF
0,Bonne continuation.,Yal na la jig te yeneeni ndam topp ci.
1,Vérifiez que votre main soit détendue le plus ...,Wóo ni sa loxo bi téguna bu baax booy bindee l...
2,"Ce renouveau politique, véritable alternative,...","Beesal gi am ci pólótig bi dana tax, ci lu wér..."
3,"Mais je ne doute point, vu la mobilisation exc...","Waaye wéddiwuma ko, soo gisee ndaje mu am dool..."
4,"Il faut que tu tiennes ta promesse, me dit dou...","Ndoomu buur si toogaat ci sama wet, wax ma ndà..."
5,Patriotes du Sénégal pour le Travail l’Ethique...,Doomi Senegaal yi fonk seen réew ngir liggéey ...
6,"C'est, du moins, ce que rapporte le quotidien ...",Loolu lañu Le quotidien L’As xibaar ci àllarba...
7,Ce qui a été bien accueilli par Serigne Mounta...,Mu doonoon lu Sëriñ Muntaqaa nangu woon a dala...
8,"Cela semble un peu intimidant d'en bas, et c'e...","Moo ngi niroog lu ragallu, te njël bu jafe ak ..."
9,Pour satisfaire aux exigences de provisionneme...,Moom rekk moo jot a dugal bit waa Pari yi ci j...


# data augmentation
actually performed worse than non-augmented training data so won't be used

In [8]:
import re

def normalize_spaces(text):
    return re.sub(r"\s+", " ", text).strip()

def remove_punctuation(text):
    return re.sub(r"[^\w\s]", "", text)

def add_random_punctuation(text):
    puncts = ["!", ".", "...", " ?"]
    return text + random.choice(puncts)

def random_word_dropout(text, p=0.1):
    words = text.split()
    if len(words) <= 2:
        return text
    kept = []
    for w in words:
        if random.random() > p:
            kept.append(w)
    if len(kept) == 0:
        kept.append(random.choice(words))
    return " ".join(kept)

def random_swap(text):
    words = text.split()
    if len(words) < 2:
        return text
    i = random.randint(0, len(words)-2)
    words[i], words[i+1] = words[i+1], words[i]
    return " ".join(words)

def accent_normalize(text):
    replacements = {"é":"e","è":"e","ê":"e","ë":"e","à":"a","â":"a","ù":"u",
                    "û":"u","î":"i","ï":"i","ô":"o","ö":"o","ç":"c"}
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text

def augment_sentence(text):
    text = normalize_spaces(text)
    ops = [
        lambda x: x,
        remove_punctuation, add_random_punctuation,
        random_swap, random_word_dropout,
        accent_normalize
    ]
    n_ops = random.randint(1, 2)
    chosen = random.sample(ops, n_ops)
    for op in chosen:
        text = op(text)
    return normalize_spaces(text)

In [9]:
aug=[]
n_aug=3 # amount of new augmented rows per original row
# for _, row in train.iterrows():
# 	fr = clean_text(row["FRENCH"])
# 	wo = clean_text(row["WOLOF"])
# 	# original sample
# 	aug.append({
# 		"FRENCH":fr,
# 		"WOLOF":wo
# 	})
# 	# augmented samples
# 	for _ in range(n_aug):
# 		aug_fr = augment_sentence(fr)
# 		aug.append({
# 			"FRENCH":aug_fr,
# 			"WOLOF":wo
# 		})

# aug_train=pd.DataFrame(aug)

# tokenization

In [10]:
fr_sentences = train["FRENCH"].astype(str).tolist()
wo_sentences = train["WOLOF"].astype(str).tolist()

In [11]:
def tokenize(x):
    return x.lower().strip().split()

fr_tokens = [tokenize(x) for x in fr_sentences]
wo_tokens = [tokenize(x) for x in wo_sentences]

# training

In [12]:
from gensim.models import FastText
fr_model = FastText(
    sentences=fr_tokens,
    vector_size=256,
    window=6,
    min_count=1,
    workers=4,
    sg=1,
    epochs=50
)
wo_model = FastText(
    sentences=wo_tokens,
    vector_size=256,
    window=6,
    min_count=1,
    workers=4,
    sg=1,
    epochs=50
)

# embedding

In [13]:
def sentence_vector(tokens, model):
    vecs=[]
    for token in tokens:
        if token in model.wv:
            vecs.append(model.wv[token])
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

# from sklearn.feature_extraction.text import TfidfVectorizer

# tfidf = TfidfVectorizer(tokenizer=tokenize,token_pattern=None)
# tfidf.fit(fr_sentences)

# def sentence_vector(tokens, model):
#     vecs = []
#     weights = []
#     for token in tokens:
#         if token in model.wv:
#             vecs.append(model.wv[token])
#             # Get TF-IDF weight (if token exists)
#             if token in tfidf.vocabulary_:
#                 idx = tfidf.vocabulary_[token]
#                 weights.append(tfidf.idf_[idx])
#             else:
#                 weights.append(1.0)
    
#     if len(vecs) == 0:
#         return np.zeros(model.vector_size)
    
#     # Weighted average by TF-IDF
#     vecs = np.array(vecs)
#     weights = np.array(weights).reshape(-1, 1)
#     weighted_vec = np.sum(vecs * weights, axis=0) / np.sum(weights)
#     return weighted_vec

In [14]:
x = np.array([sentence_vector(tokens, fr_model) for tokens in fr_tokens])
x = x / np.linalg.norm(x, axis=1, keepdims=True)

# nearest neighbor

In [15]:
from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(
    n_neighbors=5,
    metric="cosine"
)
nn.fit(x)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


# translation

In [16]:
def translate(sentence, k=3):
    tokens = tokenize(sentence)
    vec = sentence_vector(tokens, fr_model).reshape(1, -1)
    _, indices = nn.kneighbors(vec, n_neighbors=k)
    return wo_sentences[indices[0][0]]

# example

In [17]:
query = "bonjour comment allez vous"
print(translate(query))

Soo gaañoo ci jeema dimbali, kon defoo ludul yokk jafe-jafe bi.



# prediction / translation

In [18]:
def clean_text(text):
    text = str(text)

    # remove literal newlines/tabs
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = text.replace("\t", " ")

    # normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [19]:
preds=[translate(clean_text(text)) for text in test["FRENCH"]]
preds[:5]

['Rax-ci-dolli, wéq wi, ndaw si taxu ko woon a jóg, dafa doxoon ci diggante bi moo tax mu dal ko waaye wéq waa ngi jubaloon ki doon jël widéwoo yi Julius Malema (deppiteb réewum Afrique du Sud) mi di tëkku faat bakkanu deppiteb réewu Mali mi bëggoon a samp ndëndam réewam.',
 'Dëkk bu taaru bii di Casablanca la ay nappkat sos ci 10ème siècle balaa JC te ñi ko njëkoon na yor doon ay Phénicien, ay Romain ak Mérénide yi, mu doonoon port bu am mug bu tuddoon Anfa.\n',
 'Su fekkee ne soriwante bi mënul nekk ci dëkkuwaay yeek daara yu kawe yu bokk-moomeel yi, ndax nit ñu bare ak ñàkk jumtukaay yu mat, takk mëddukaay (mask) yi itam génnu ci\xa0: néewaayu saytukat yi ci iniwersite yi, càgganteek askan week ñàkk a matale ñaw ay mëddukaay (mask) ak wasaare leen ba tax askan wi ci boppam, ñooy góor-góorlu di ko def, ni ki ndongo.',
 "Waaye yaakaarnañu ni amnañu yoon xam ko ni nga ame yoon xam njeexital yu nekk ci tann tux sigaret' Jangalekat Czeisler yokk ko ci.\n",
 'Duggsi gu tar bi dina des ci 

In [20]:
subs_df=pd.DataFrame({
  "ID":test["ID"],
  "Target":preds,
})
# subs_df["Target"] = subs_df["Target"].str.replace("\"", " ", regex=False)

In [21]:
subs_df.head()
# subs_df.shape

,ID,Target
0,0,"Rax-ci-dolli, wéq wi, ndaw si taxu ko woon a j..."
1,1,Dëkk bu taaru bii di Casablanca la ay nappkat ...
2,2,Su fekkee ne soriwante bi mënul nekk ci dëkkuw...
3,3,Waaye yaakaarnañu ni amnañu yoon xam ko ni nga...
4,4,"Duggsi gu tar bi dina des ci mboor mi, ni ñu k..."


In [22]:
subs_df.to_csv("submission.csv",index=False)